1. PROJECT TITLE - PHONEPE TRANSACTION INSIGHTS.

2. PROJECT SUMMARY -

    This project focuses on building an end-to-end data analytics pipeline to extract, process, and visualize PhonePe data. SQL was used to query and aggregate data across transactions, users, and insurance domains, while Python was used for data transformation and exploratory analysis.

    The insights are presented through an interactive Streamlit dashboard that enables dynamic exploration of trends across states, districts, and pincodes. The project highlights key growth patterns, regional variations, and engagement metrics, providing a comprehensive view of the digital payments ecosystem.

3. PROBLEM STATEMENT -

    With the increasing reliance on digital payment systems like PhonePe, understanding the dynamics of transactions, user engagement, and insurance-related data is crucial for improving services and targeting users effectively. This project aims to analyze and visualize aggregated values of payment categories, create maps for total values at state and district levels, and identify top-performing states, districts, and pin codes.

4. Connecting Python to the database in MySQL -

In [2]:
# Importing the library to connect
import mysql.connector

# Initialising the database connection
conn = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'Tatakae#1996',
    database = 'phonepe'
    )

cursor = conn.cursor()
print('Connected Successfully')

Connected Successfully


5. Inserting data for aggregated-->insurance-->state.

In [3]:
# Importing the necessary libraries
import os
import json

In [4]:
# Base directory
base_path_ins = os.path.join('..', 'unclean_data', 'data', 'aggregated', 'insurance', 'country', 'india', 'state')

# Traverse states
for state_ins in os.listdir(base_path_ins):
    state_path_ins = os.path.join(base_path_ins, state_ins)

    if not os.path.isdir(state_path_ins):
        continue

    # Traverse years
    for year_ins in os.listdir(state_path_ins):
        year_path_ins = os.path.join(state_path_ins, year_ins)

        if not os.path.isdir(year_path_ins):
            continue

        # Traverse quarter files
        for file_ins in os.listdir(year_path_ins):
            if not file_ins.endswith('.json'):
                continue
            
            quarter_ins = int(file_ins.replace('.json', ''))
            file_path_ins = os.path.join(year_path_ins, file_ins)

            with open (file_path_ins, 'r') as f:
                data_ins = json.load(f)

            # Safe extraction
            if 'data' not in data_ins or data_ins['data'] is None:
                continue

            transactions_ins = data_ins['data'].get('transactionData', [])

            for item_ins in transactions_ins:
                transaction_name = item_ins.get('name')

                for inst_ins in item_ins.get('paymentInstruments', []):
                    transaction_type = inst_ins.get('type')
                    transaction_count = inst_ins.get('count')
                    transaction_amount = inst_ins.get('amount')

                    # Insert query
                    query_ins = """
                            insert into aggregated_insurance_state
                            (state, year, quarter, transaction_name, transaction_type, transaction_count, transaction_amount)
                            values (%s, %s, %s, %s, %s, %s, %s)"""
                    
                    values_ins = (
                        state_ins, int(year_ins), quarter_ins,
                        transaction_name,
                        transaction_type,
                        transaction_count,
                        transaction_amount
                    )

                    cursor.execute(query_ins, values_ins)

print('Aggregated insurance state data inserted successfully')

Aggregated insurance state data inserted successfully


6. Inserting data for aggregated-->transaction-->state.

In [5]:
# Base path for aggregated-->transaction-->state
base_path_txn = os.path.join('..', 'unclean_data', 'data', 'aggregated', 'transaction', 'country', 'india', 'state')

# Traverse states
for state_txn in os.listdir(base_path_txn):
    state_path_txn = os.path.join(base_path_txn, state_txn)

    if not os.path.isdir(state_path_txn):
        continue

    # Traverse years
    for year_txn in os.listdir(state_path_txn):
        year_path_txn = os.path.join(state_path_txn, year_txn)

        if not os.path.isdir(year_path_txn):
            continue

        # Traverse quarters
        for file_txn in os.listdir(year_path_txn):
            if not file_txn.endswith('.json'):
                continue

            quarter_txn = int(file_txn.replace('.json', ''))
            file_path_txn = os.path.join(year_path_txn, file_txn)

            with open (file_path_txn, 'r') as f:
                data_txn = json.load(f)

            # Safe extraction
            if 'data' not in data_txn or data_txn['data'] is None:
                continue

            transactions_txn = data_txn['data'].get('transactionData', [])

            for item_txn in transactions_txn:
                transaction_name = item_txn.get('name')

                for inst_txn in item_txn.get('paymentInstruments', []):
                    transaction_type = inst_txn.get('type')
                    transaction_count = inst_txn.get('count')
                    transaction_amount = inst_txn.get('amount')

                    # Insert query
                    query_txn = """
                                insert into aggregated_transaction_state
                            (state, year, quarter, transaction_name, transaction_type, transaction_count, transaction_amount)
                            values (%s, %s, %s, %s, %s, %s, %s)"""
                    
                    values_txn = (state_txn, year_txn, quarter_txn,
                                  transaction_name, transaction_type,
                                  transaction_count, transaction_amount)
                    
                    cursor.execute(query_txn, values_txn)

print('Aggregated transaction state data inserted successfully')

Aggregated transaction state data inserted successfully


7. Inserting data for aggregated-->user-->state.

In [8]:
# Base path for aggregated-->user-->state
base_path_user = os.path.join('..', 'unclean_data', 'data', 'aggregated', 'user', 'country', 'india', 'state')

# Traverse states
for state_user in os.listdir(base_path_user):
    state_path_user = os.path.join(base_path_user, state_user)

    if not os.path.isdir(state_path_user):
        continue

    # Traverse years
    for year_user in os.listdir(state_path_user):
        year_path_user = os.path.join(state_path_user, year_user)

        if not os.path.isdir(year_path_user):
            continue

        # Traverse quarters
        for file_user in os.listdir(year_path_user):
            if not file_user.endswith('.json'):
                continue

            quarter_user = int(file_user.replace('.json', ''))
            file_path_user = os.path.join(year_path_user, file_user)

            with open(file_path_user, 'r') as f:
                data_user = json.load(f)

            # Safe extraction
            if 'data' not in data_user or data_user['data'] is None:
                continue

            # 1. Aggregated (state-level)
            aggregated_user = data_user['data'].get('aggregated', {})

            registered_users = aggregated_user.get('registeredUsers')
            app_opens = aggregated_user.get('appOpens')

            query_user_agg = """
                INSERT INTO aggregated_user_state
                (state, year, quarter, registered_users, app_opens)
                VALUES (%s, %s, %s, %s, %s)
            """

            values_user_agg = (state_user, int(year_user), quarter_user,
                               registered_users, app_opens)

            cursor.execute(query_user_agg, values_user_agg)

            # 2. Device-level
            users_device_list = data_user['data'].get('usersByDevice') or []

            for device_user in users_device_list:
                brand_user = device_user.get('brand')
                user_count_user = device_user.get('count')
                percentage_user = device_user.get('percentage')

                query_user_device = """
                    INSERT INTO aggregated_user_device_state
                    (state, year, quarter, brand, user_count, percentage)
                    VALUES (%s, %s, %s, %s, %s, %s)
                """

                values_user_device = (state_user, int(year_user), quarter_user,
                                      brand_user, user_count_user, percentage_user)

                cursor.execute(query_user_device, values_user_device)

print('Aggregated user state data inserted successfully')

Aggregated user state data inserted successfully


In [9]:
# Commit
conn.commit()

# Close
cursor.close()
conn.close()